In [3]:
import sys
import os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
# ==============================================================================
# FILE 11: FINE-TUNE ON EXTERNAL DATA
# ==============================================================================

print("="*80)
print("🔧 FILE 11: FINE-TUNE CLASSIFICATION ON EXTERNAL DATA")
print("="*80)

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch. utils.data import Dataset, DataLoader
import cv2
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import pandas as pd
import timm
import json

# Mount

GDRIVE_PATH = "d:/DoAn_DaLieu"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ==============================================================================
# 1. LOAD MODEL HIỆN TẠI
# ==============================================================================

print("\n" + "="*80)
print("📦 LOADING CURRENT MODEL")
print("="*80)

# Load checkpoint
checkpoint_path = os.path.join(GDRIVE_PATH, "3_Checkpoints/06_classification_complete.json")
with open(checkpoint_path, 'r') as f:
    cls_checkpoint = json.load(f)

# Load model weights
cls_state = torch.load(cls_checkpoint['paths']['best_model'], map_location=device)
num_classes = cls_checkpoint['config']['num_classes']
class_to_idx = cls_state['class_to_idx']
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Model architecture (copy từ File 06)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(in_channels, reduction)
        self.spatial_att = SpatialAttention()

    def forward(self, x):
        return x * self. channel_att(x) * self.spatial_att(x * self.channel_att(x))

class EfficientNetWithAttention(nn.Module):
    def __init__(self, num_classes, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.feature_dim = self.backbone.num_features
        self.attention = CBAM(self.feature_dim, reduction=16)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn. Dropout(0.3),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone. forward_features(x)
        features_att = self.attention(features)
        features_pooled = self.global_pool(features_att).flatten(1)
        return self.classifier(features_pooled)

# Load model
model = EfficientNetWithAttention(num_classes=num_classes, pretrained=False)
model.load_state_dict(cls_state['model_state_dict'])
model = model.to(device)

print("✅ Model loaded")
print(f"   Classes: {list(idx_to_class.values())}")

# ==============================================================================
# 2. EXTERNAL DATASET
# ==============================================================================

class ExternalDataset(Dataset):
    def __init__(self, csv_path, data_root, transform=None, use_clahe=False):
        self.df = pd.read_csv(csv_path)
        self.data_root = data_root
        self. transform = transform
        self.use_clahe = use_clahe
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        label = row['ground_truth']

        # Load image từ Data_test/{class}/{image}
        img_path = os. path.join(self.data_root, label, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Optional CLAHE
        if self.use_clahe:
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            lab[:,:,0] = clahe. apply(lab[:,:,0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)

        label_idx = self.class_to_idx[label]

        return image, label_idx

# Transforms - STRONG augmentation cho fine-tuning
train_transform = transforms.Compose([
    transforms. Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Datasets
DATA_TEST_PATH = os.path.join(GDRIVE_PATH, "1_Data/Data_test")
train_csv = os.path.join(GDRIVE_PATH, "7_External_Test/external_train.csv")
test_csv = os.path.join(GDRIVE_PATH, "7_External_Test/external_test_split.csv")

train_dataset = ExternalDataset(train_csv, DATA_TEST_PATH, transform=train_transform, use_clahe=False)
val_dataset = ExternalDataset(test_csv, DATA_TEST_PATH, transform=val_transform, use_clahe=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"\n✅ Datasets ready")
print(f"   Train: {len(train_dataset)}")
print(f"   Val:     {len(val_dataset)}")

# ==============================================================================
# 3. FINE-TUNE SETUP
# ==============================================================================

print("\n" + "="*80)
print("🔧 FINE-TUNE SETUP")
print("="*80)

# FREEZE backbone (chỉ train classifier)
for param in model.backbone.parameters():
    param.requires_grad = False

for param in model.attention.parameters():
    param.requires_grad = False

# UNFREEZE classifier
for param in model.classifier. parameters():
    param.requires_grad = True

print("✅ Backbone + Attention:  FROZEN")
print("✅ Classifier: TRAINABLE")

# Optimizer - LEARNING RATE CỰC THẤP
optimizer = optim. Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,  # ← CỰC THẤP (gốc:  1e-3)
    weight_decay=1e-4
)

# Loss
criterion = nn.CrossEntropyLoss()

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

print(f"✅ Optimizer:  Adam (lr=1e-5)")
print(f"✅ Loss: CrossEntropyLoss")

# ==============================================================================
# 4. FINE-TUNE
# ==============================================================================

print("\n" + "="*80)
print("🚀 FINE-TUNING")
print("="*80)

NUM_EPOCHS = 10  # Ít epochs (tránh overfit)
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)

    # Train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    train_acc = 100. * train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch. no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs. max(1)
            val_total += labels.size(0)
            val_correct += predicted. eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total

    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:     {val_loss/len(val_loader):.4f} | Val Acc:    {val_acc:.2f}%")

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = os.path.join(GDRIVE_PATH, "3_Checkpoints/06_classification_finetuned.pth")
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_to_idx': class_to_idx,
            'val_acc': val_acc,
            'epoch':  epoch
        }, save_path)
        print(f"✅ Saved best model (val_acc: {val_acc:.2f}%)")

    scheduler.step()

print("\n" + "="*80)
print(f"✅ FINE-TUNING COMPLETE!")
print(f"   Best Val Acc: {best_val_acc:.2f}%")
print("="*80)

🔧 FILE 11: FINE-TUNE CLASSIFICATION ON EXTERNAL DATA
Device: cpu

📦 LOADING CURRENT MODEL
✅ Model loaded
   Classes: ['NV', 'MEL', 'BKL', 'DF', 'AKIEC', 'BCC', 'VASC']

✅ Datasets ready
   Train: 50
   Val:     17

🔧 FINE-TUNE SETUP
✅ Backbone + Attention:  FROZEN
✅ Classifier: TRAINABLE
✅ Optimizer:  Adam (lr=1e-5)
✅ Loss: CrossEntropyLoss

🚀 FINE-TUNING

Epoch 1/10
----------------------------------------


Training: 100%|██████████| 7/7 [00:09<00:00,  1.34s/it]


Train Loss: 6.9099 | Train Acc: 4.00%
Val Loss:     5.2127 | Val Acc:    41.18%
✅ Saved best model (val_acc: 41.18%)

Epoch 2/10
----------------------------------------


Training: 100%|██████████| 7/7 [00:06<00:00,  1.12it/s]


Train Loss: 7.7111 | Train Acc: 12.00%
Val Loss:     8.0094 | Val Acc:    35.29%

Epoch 3/10
----------------------------------------


...

Train Loss: 7.5206 | Train Acc: 12.00%
Val Loss:     9.9453 | Val Acc:    23.53%

Epoch 9/10
----------------------------------------


Training: 100%|██████████| 7/7 [00:05<00:00,  1.26it/s]


Train Loss: 7.6373 | Train Acc: 12.00%
Val Loss:     9.9884 | Val Acc:    23.53%

Epoch 10/10
----------------------------------------


Training: 100%|██████████| 7/7 [00:05<00:00,  1.27it/s]


Train Loss: 6.7480 | Train Acc: 10.00%
Val Loss:     8.9521 | Val Acc:    23.53%

✅ FINE-TUNING COMPLETE!
   Best Val Acc: 41.18%


In [6]:
# ==============================================================================
# 5. TEST FINETUNED MODEL ON FULL EXTERNAL SET
# ==============================================================================

print("\n" + "="*80)
print("🧪 TESTING FINETUNED MODEL")
print("="*80)

# Load finetuned weights
finetuned_path = os.path.join(GDRIVE_PATH, "3_Checkpoints/06_classification_finetuned.pth")
finetuned_state = torch.load(finetuned_path, map_location=device)

model.load_state_dict(finetuned_state['model_state_dict'])
model = model.to(device).eval()

print(f"✅ Loaded finetuned model (best val acc: {finetuned_state['val_acc']:.2f}%)")

# Test trên TOÀN BỘ 67 ảnh external
full_csv = os.path.join(GDRIVE_PATH, "7_External_Test/external_test.csv")
full_dataset = ExternalDataset(full_csv, DATA_TEST_PATH, transform=val_transform, use_clahe=False)
full_loader = DataLoader(full_dataset, batch_size=8, shuffle=False)

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(full_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels. numpy())

# Calculate accuracy
correct = sum([1 for p, l in zip(all_preds, all_labels) if p == l])
accuracy = 100. * correct / len(all_labels)

print(f"\n" + "="*80)
print(f"📊 FINETUNED MODEL RESULTS")
print(f"="*80)
print(f"\n✅ Accuracy: {accuracy:.1f}% ({correct}/{len(all_labels)})")

# Per-class
df_full = pd.read_csv(full_csv)
for cls in sorted(df_full['ground_truth'].unique()):
    cls_idx = class_to_idx[cls]
    cls_mask = [l == cls_idx for l in all_labels]
    cls_correct = sum([p == l for p, l, m in zip(all_preds, all_labels, cls_mask) if m])
    cls_total = sum(cls_mask)
    cls_acc = 100. * cls_correct / cls_total if cls_total > 0 else 0
    print(f"   {cls:6s}: {cls_acc: 5.1f}% ({cls_correct}/{cls_total})")

print("\n" + "="*80)
if accuracy >= 70:
    print("🎉 FINETUNED MODEL ACHIEVES TARGET (>= 70%)!")
elif accuracy >= 60:
    print("⚠️ Close to target, consider more epochs or data")
else:
    print("❌ Need more improvements")


🧪 TESTING FINETUNED MODEL
✅ Loaded finetuned model (best val acc: 41.18%)


Testing: 100%|██████████| 9/9 [00:07<00:00,  1.26it/s]



📊 FINETUNED MODEL RESULTS

✅ Accuracy: 25.4% (17/67)
   AKIEC :  16.7% (2/12)
   BCC   :  20.0% (3/15)
   BKL   :  25.0% (2/8)
   DF    :  45.5% (5/11)
   MEL   :  13.3% (2/15)
   NV    :  66.7% (2/3)
   VASC  :  33.3% (1/3)

❌ Need more improvements
